In [1]:
import os
import sys
import pandas as pd
import numpy as np
import yaml
import optuna
import joblib

optuna.logging.set_verbosity(optuna.logging.WARNING)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(PROJECT_ROOT, "src"))
from harmonization import HARMONIZED_FEATURE_COLS
from modeling import make_objective, build_model_pipeline, tune_threshold, bootstrap_ci, evaluate_at_threshold
from shap_utils import compute_shap_values, get_top_features

with open(os.path.join(PROJECT_ROOT, "config", "config.yaml")) as f:
    config = yaml.safe_load(f)

SEED = config["seed"]
EXTERNAL_DIR = os.path.join(PROJECT_ROOT, config["paths"]["external_dir"])
MODELS_ROOT = os.path.join(PROJECT_ROOT, config["paths"]["models_dir"])
TABLES_DIR = os.path.join(PROJECT_ROOT, config["paths"]["tables_dir"])
N_BOOTSTRAP = config["bootstrap"]["n_resamples"]

pk_h = pd.read_csv(os.path.join(EXTERNAL_DIR, "pakistan_harmonized.csv"))
ar_h = pd.read_csv(os.path.join(EXTERNAL_DIR, "arab5_harmonized.csv"))
print("Pakistan:", pk_h.shape, "| Arab5:", ar_h.shape)

Pakistan: (14002, 7) | Arab5: (1498, 8)


In [2]:
def train_and_validate_externally(source_name, source_df, target_name, target_df,
                                   n_trials=15, seed=SEED):
    X_source = source_df[HARMONIZED_FEATURE_COLS]
    y_source = source_df["y_depression"]

    train_A, cal_A = train_test_split(source_df, test_size=0.2, stratify=y_source, random_state=seed)

    # Light Optuna tuning -- no group-aware CV needed here: with only 5 binary
    # features (32 possible profiles), every fold will necessarily contain
    # repeated profiles regardless of split strategy. That's a structural
    # property of this coarse feature space, not a data-quality artifact to
    # correct for (unlike the earlier Pakistan/Postnatal duplicate-row issue).
    objective = make_objective("rf", train_A[HARMONIZED_FEATURE_COLS], train_A["y_depression"],
                                numeric_cols=[], categorical_cols=HARMONIZED_FEATURE_COLS,
                                seed=seed, n_splits=5, groups=None)
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    print(f"[{source_name} -> {target_name}] Best CV AUC (within {source_name}): {study.best_value:.4f}")

    pipe = build_model_pipeline("rf", study.best_params, numeric_cols=[],
                                 categorical_cols=HARMONIZED_FEATURE_COLS, seed=seed)
    pipe.fit(train_A[HARMONIZED_FEATURE_COLS], train_A["y_depression"])

    proba_cal = pipe.predict_proba(cal_A[HARMONIZED_FEATURE_COLS])[:, 1]
    thr, macro_f1 = tune_threshold(cal_A["y_depression"], proba_cal)
    cal_auc = bootstrap_ci(cal_A["y_depression"], proba_cal, n_resamples=N_BOOTSTRAP, seed=seed)
    print(f"[{source_name} internal] threshold={thr:.2f}, calibration AUC={cal_auc[0]:.4f} [{cal_auc[1]:.4f}, {cal_auc[2]:.4f}]")

    # Freeze model + threshold, apply WITHOUT modification to the full target country
    X_target = target_df[HARMONIZED_FEATURE_COLS]
    y_target = target_df["y_depression"]
    proba_target = pipe.predict_proba(X_target)[:, 1]

    metrics = evaluate_at_threshold(y_target, proba_target, thr)
    auc_point, auc_lo, auc_hi = bootstrap_ci(y_target, proba_target, n_resamples=N_BOOTSTRAP, seed=seed)
    print(f"[{source_name} -> {target_name}] EXTERNAL: Acc={metrics['accuracy']:.4f}  "
          f"MacroF1={metrics['macro_f1']:.4f}  AUC={metrics['auc_roc']:.4f} [{auc_lo:.4f}, {auc_hi:.4f}]")

    return {
        "direction": f"{source_name}_to_{target_name}",
        "internal_cv_auc": round(study.best_value, 4),
        "internal_calibration_auc": round(cal_auc[0], 4),
        "external_auc": round(metrics["auc_roc"], 4),
        "external_auc_ci_lower": round(auc_lo, 4), "external_auc_ci_upper": round(auc_hi, 4),
        "external_accuracy": round(metrics["accuracy"], 4),
        "external_macro_f1": round(metrics["macro_f1"], 4),
        "threshold_used": round(thr, 2),
        "n_source_train": len(train_A), "n_target_test": len(target_df),
    }, pipe

In [3]:
from sklearn.model_selection import train_test_split

In [4]:
result_pk_to_ar, model_pk = train_and_validate_externally("pakistan", pk_h, "arab5", ar_h)

[pakistan -> arab5] Best CV AUC (within pakistan): 0.5230
[pakistan internal] threshold=0.50, calibration AUC=0.5171 [0.4954, 0.5393]
[pakistan -> arab5] EXTERNAL: Acc=0.4887  MacroF1=0.4870  AUC=0.4802 [0.4510, 0.5104]


In [5]:
result_ar_to_pk, model_ar = train_and_validate_externally("arab5", ar_h, "pakistan", pk_h)

[arab5 -> pakistan] Best CV AUC (within arab5): 0.5695
[arab5 internal] threshold=0.52, calibration AUC=0.5726 [0.5092, 0.6348]
[arab5 -> pakistan] EXTERNAL: Acc=0.4839  MacroF1=0.4762  AUC=0.4876 [0.4776, 0.4977]


In [6]:
summary_df = pd.DataFrame([result_pk_to_ar, result_ar_to_pk])
print(summary_df)
summary_df.to_csv(os.path.join(TABLES_DIR, "cross_country_external_validation.csv"), index=False)

           direction  internal_cv_auc  internal_calibration_auc  external_auc  \
0  pakistan_to_arab5           0.5230                    0.5171        0.4802   
1  arab5_to_pakistan           0.5695                    0.5726        0.4876   

   external_auc_ci_lower  external_auc_ci_upper  external_accuracy  \
0                 0.4510                 0.5104             0.4887   
1                 0.4776                 0.4977             0.4839   

   external_macro_f1  threshold_used  n_source_train  n_target_test  
0             0.4870            0.50           11201           1498  
1             0.4762            0.52            1198          14002  


In [7]:
shap_pk_model, names_pk, _, _ = compute_shap_values(model_pk, pk_h[HARMONIZED_FEATURE_COLS], sample_size=500, seed=SEED)
shap_ar_model, names_ar, _, _ = compute_shap_values(model_ar, ar_h[HARMONIZED_FEATURE_COLS], sample_size=500, seed=SEED)

top_pk = get_top_features(shap_pk_model, names_pk, n=10)
top_ar = get_top_features(shap_ar_model, names_ar, n=10)

print("Pakistan-trained model, top features:")
print(top_pk)
print("\nArab5-trained model, top features:")
print(top_ar)

top_pk.to_csv(os.path.join(TABLES_DIR, "external_validation_shap_pakistan_model.csv"), index=False)
top_ar.to_csv(os.path.join(TABLES_DIR, "external_validation_shap_arab5_model.csv"), index=False)

Pakistan-trained model, top features:
                                  feature  mean_abs_shap
0                  cat__employed_employed       0.006491
1              cat__employed_not_employed       0.006213
2            cat__financial_strain_strain       0.004494
3         cat__financial_strain_no_strain       0.004381
4                 cat__age_group_under_35       0.002854
5                  cat__age_group_35_plus       0.002502
6    cat__first_pregnancy_first_pregnancy       0.001905
7          cat__first_pregnancy_not_first       0.001712
8    cat__education_group_above_secondary       0.001157
9  cat__education_group_secondary_or_less       0.001023

Arab5-trained model, top features:
                                  feature  mean_abs_shap
0         cat__financial_strain_no_strain       0.033120
1            cat__financial_strain_strain       0.029383
2              cat__employed_not_employed       0.014556
3                  cat__employed_employed       0.012893
4          cat